In [0]:
# Core LLM and Embedding libraries
%pip install -q transformers accelerate sentencepiece safetensors sentence-transformers

# Search and Vector DB libraries
%pip install -q faiss-cpu rank-bm25 

# Data handling
%pip install -q pandas numpy

# Optional: if you are using .env files for HF_TOKEN
%pip install -q python-dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install -q peft

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from dotenv import load_dotenv
import os
load_dotenv()
mail_address = os.getenv("EMAIL")

In [0]:
BASE_DIR = f"/Workspace/Users/{mail_address}/nyaya_deepam"
INDEX_FULL_DIR = f"{BASE_DIR}/index/full"
INDEX_ATOMIC_DIR = f"{BASE_DIR}/index/no_composite"

In [0]:
# =========================
# BLOCK 1: BUILD + PERSIST
# =========================

# %pip install -q pandas sentence-transformers faiss-cpu rank-bm25 python-dotenv
# dbutils.library.restartPython()

import os
import re
import json
import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# ---------- config ----------
load_dotenv()

EMAIL = os.getenv("EMAIL")  # set this in your env/.env
HF_TOKEN = os.getenv("HF_TOKEN")

assert EMAIL, "Set EMAIL in env/.env"
assert HF_TOKEN, "Set HF_TOKEN in env/.env"

PROJECT = "nyaya_deepam"
BASE_ROOT = f"/Workspace/Users/{EMAIL}/{PROJECT}"
RAW_DIR = f"{BASE_ROOT}/raw"
ARTIFACT_DIR = f"{BASE_ROOT}/artifacts"
FULL_DIR = f"{ARTIFACT_DIR}/full"
ATOMIC_DIR = f"{ARTIFACT_DIR}/atomic"

TABLE_FULL = "nyaya_deepam_legal_rag"
TABLE_ATOMIC = "nyaya_deepam_legal_rag_atomic"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"

os.makedirs(FULL_DIR, exist_ok=True)
os.makedirs(ATOMIC_DIR, exist_ok=True)

# ---------- helpers ----------
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

def flatten(df, source):
    rows = []
    for _, r in df.iterrows():
        meta = r.get("metadata", {}) or {}
        rows.append({
            "chunk_id": str(r.get("chunk_id", "")),
            "text": str(r.get("text", "")),
            "act_name": str(meta.get("act_name", "")),
            "chapter": str(meta.get("chapter", "")),
            "section": str(meta.get("section", meta.get("ipc_section", meta.get("bns_section_subsection", "")))),
            "section_name": str(meta.get("section_name", meta.get("subject", ""))),
            "source": str(source),
        })
    return pd.DataFrame(rows)

def normalize_section(sec):
    sec = str(sec).strip().upper()
    sec = re.sub(r"\s+", "", sec)
    sec = re.sub(r"\(.*?\)", "", sec)  # 1(1) -> 1
    return sec

def build_multi_lookup(df, key_col):
    lookup = {}
    for _, row in df.iterrows():
        key = row[key_col]
        lookup.setdefault(key, []).append(row.to_dict())
    return lookup

def choose_best_match(candidates, target_section):
    if not candidates:
        return {}
    target_section = str(target_section).strip().upper()

    # exact raw match first
    for c in candidates:
        if str(c.get("section", "")).strip().upper() == target_section:
            return c

    # otherwise longest text
    candidates = sorted(candidates, key=lambda x: len(str(x.get("text", ""))), reverse=True)
    return candidates[0]

def build_composite_chunks(bns_flat, ipc_flat, map_flat):
    bns = bns_flat.copy()
    ipc = ipc_flat.copy()
    mp = map_flat.copy()

    bns["norm_section"] = bns["section"].apply(normalize_section)
    ipc["norm_section"] = ipc["section"].apply(normalize_section)

    # mapping table in your current format
    mp["norm_bns"] = mp["section"].apply(normalize_section)
    mp["norm_ipc"] = mp["section_name"].apply(normalize_section)

    bns_lookup = build_multi_lookup(bns, "norm_section")
    ipc_lookup = build_multi_lookup(ipc, "norm_section")

    composite_rows = []
    for _, row in mp.iterrows():
        bns_raw = row.get("section", "")
        ipc_raw = row.get("section_name", "")

        bns_data = choose_best_match(bns_lookup.get(row.get("norm_bns", ""), []), bns_raw)
        ipc_data = choose_best_match(ipc_lookup.get(row.get("norm_ipc", ""), []), ipc_raw)

        bns_text = str(bns_data.get("text", "")).strip()
        ipc_text = str(ipc_data.get("text", "")).strip()
        mapping_text = str(row.get("text", "")).strip()

        combined_text = f"""BNS Section: {bns_raw}
{bns_text}

IPC Section: {ipc_raw}
{ipc_text}

Mapping and Changes:
{mapping_text}""".strip()

        composite_rows.append({
            "chunk_id": f"COMPOSITE_{row.get('norm_bns','')}_{row.get('norm_ipc','')}",
            "text": combined_text,
            "act_name": "BNS_IPC_COMPOSITE",
            "chapter": "",
            "section": str(bns_raw),
            "section_name": str(ipc_raw),
            "source": "composite_mapping",
        })

    return pd.DataFrame(composite_rows)

def save_bundle(index_dir, corpus_df, embed_model):
    os.makedirs(index_dir, exist_ok=True)

    texts = corpus_df["text"].fillna("").astype(str).tolist()
    passages = ["passage: " + t for t in texts]

    embeddings = embed_model.encode(
        passages,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(np.array(embeddings, dtype=np.float32))

    faiss.write_index(index, f"{index_dir}/corpus.faiss")

    clean_cols = ["chunk_id", "text", "act_name", "chapter", "section", "section_name", "source"]
    clean_df = pd.DataFrame({
        col: corpus_df[col].astype(str).tolist()
        for col in clean_cols
    })

    with open(f"{index_dir}/chunks.jsonl", "w", encoding="utf-8") as f:
        for row in clean_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    clean_df.to_csv(f"{index_dir}/chunks.csv", index=False, encoding="utf-8")

    with open(f"{index_dir}/manifest.json", "w", encoding="utf-8") as f:
        json.dump({
            "embedding_model": EMBED_MODEL_NAME,
            "num_chunks": int(len(clean_df)),
            "dim": int(index.d),
        }, f, indent=2)

    print(f"Saved bundle: {index_dir}")
    print("FAISS:", index.ntotal, "| Rows:", len(clean_df))

# ---------- load raw files ----------
df_bns = load_jsonl(f"{RAW_DIR}/bns_rag.jsonl")
df_ipc = load_jsonl(f"{RAW_DIR}/ipc_rag.jsonl")
df_map = load_jsonl(f"{RAW_DIR}/bns_ipc_rag.jsonl")

bns_flat = flatten(df_bns, "BNS_2023")
ipc_flat = flatten(df_ipc, "IPC_1860")
map_flat = flatten(df_map, "BNS_IPC_MAPPING")

composite_df = build_composite_chunks(bns_flat, ipc_flat, map_flat)

# ---------- final corpora ----------
full_corpus_df = pd.concat([bns_flat, ipc_flat, map_flat, composite_df], ignore_index=True).fillna("").astype(str)
atomic_corpus_df = full_corpus_df[full_corpus_df["source"] != "composite_mapping"].copy().reset_index(drop=True)

print("FULL corpus:")
print(full_corpus_df["source"].value_counts())
print("\nATOMIC corpus:")
print(atomic_corpus_df["source"].value_counts())

# ---------- save delta tables ----------
spark.createDataFrame(full_corpus_df).write.format("delta").mode("overwrite").saveAsTable(TABLE_FULL)
spark.createDataFrame(atomic_corpus_df).write.format("delta").mode("overwrite").saveAsTable(TABLE_ATOMIC)

print(f"Saved Delta tables: {TABLE_FULL}, {TABLE_ATOMIC}")

# ---------- login HF ----------
from huggingface_hub import login
login(HF_TOKEN)

# ---------- load embed model ----------
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

# ---------- save persistent bundles ----------
save_bundle(FULL_DIR, full_corpus_df, embed_model)
save_bundle(ATOMIC_DIR, atomic_corpus_df, embed_model)

print("\nDone.")
print("Persistent folders:")
print(" FULL  :", FULL_DIR)
print(" ATOMIC:", ATOMIC_DIR)

FULL corpus:
source
BNS_IPC_MAPPING      530
composite_mapping    530
BNS_2023             358
IPC_1860             161
Name: count, dtype: int64

ATOMIC corpus:
source
BNS_IPC_MAPPING    530
BNS_2023           358
IPC_1860           161
Name: count, dtype: int64


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Saved Delta tables: nyaya_deepam_legal_rag, nyaya_deepam_legal_rag_atomic


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Saved bundle: /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/full
FAISS: 1579 | Rows: 1579


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Saved bundle: /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/atomic
FAISS: 1049 | Rows: 1049

Done.
Persistent folders:
 FULL  : /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/full
 ATOMIC: /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/atomic


In [0]:
# =========================
# BLOCK 2: INFERENCE ONLY
# =========================

# %pip install -q sentence-transformers faiss-cpu rank-bm25 "transformers>=4.37.0" accelerate sentencepiece safetensors python-dotenv
# dbutils.library.restartPython()

import os
import json
import faiss
import torch
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- config ----------
load_dotenv()

EMAIL = os.getenv("EMAIL")
HF_TOKEN = os.getenv("HF_TOKEN")

assert EMAIL, "Set EMAIL in env/.env"
assert HF_TOKEN, "Set HF_TOKEN in env/.env"

PROJECT = "nyaya_deepam"
BASE_ROOT = f"/Workspace/Users/{EMAIL}/{PROJECT}"
FULL_DIR = f"{BASE_ROOT}/artifacts/full"
ATOMIC_DIR = f"{BASE_ROOT}/artifacts/atomic"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# ---------- helpers ----------
def load_index_bundle(index_dir):
    index = faiss.read_index(f"{index_dir}/corpus.faiss")

    rows = []
    with open(f"{index_dir}/chunks.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    chunks_df = pd.DataFrame(rows).reset_index(drop=True)

    if index.ntotal != len(chunks_df):
        raise ValueError(f"Mismatch in {index_dir}: FAISS={index.ntotal}, rows={len(chunks_df)}")

    texts = chunks_df["text"].fillna("").astype(str).tolist()
    bm25 = BM25Okapi([t.lower().split() for t in texts])

    print(f"Loaded {index_dir}")
    print("FAISS:", index.ntotal, "| Rows:", len(chunks_df))

    return {
        "index": index,
        "chunks_df": chunks_df,
        "bm25": bm25,
    }

def choose_mode(query: str) -> str:
    q = query.lower()
    keywords = ["equivalent", "mapping", "mapped", "difference", "changed", "compare", "ipc", "bns"]
    return "full" if any(k in q for k in keywords) else "atomic"

# ---------- login + models ----------
login(HF_TOKEN)

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto"
)

bundle_full = load_index_bundle(FULL_DIR)
bundle_atomic = load_index_bundle(ATOMIC_DIR)

def embed_query(query):
    return embed_model.encode(["query: " + query], normalize_embeddings=True)

def hybrid_search(query, top_k=5, alpha=0.65, mode="full"):
    bundle = bundle_full if mode == "full" else bundle_atomic
    index = bundle["index"]
    chunks_df = bundle["chunks_df"]
    bm25 = bundle["bm25"]

    q_vec = embed_query(query)
    vec_scores, vec_idx = index.search(np.array(q_vec, dtype=np.float32), top_k * 3)

    vec_scores = vec_scores[0]
    vec_idx = vec_idx[0]

    if len(vec_scores) > 0:
        vmin, vmax = vec_scores.min(), vec_scores.max()
        vec_scores = (vec_scores - vmin) / (vmax - vmin + 1e-8)

    q_tokens = query.lower().split()
    bm25_scores = np.array(bm25.get_scores(q_tokens), dtype=np.float32)

    if len(bm25_scores) > 0:
        bmin, bmax = bm25_scores.min(), bm25_scores.max()
        bm25_scores = (bm25_scores - bmin) / (bmax - bmin + 1e-8)

    combined = {}

    for i, idx in enumerate(vec_idx):
        idx = int(idx)
        if 0 <= idx < len(chunks_df):
            combined[idx] = combined.get(idx, 0.0) + alpha * float(vec_scores[i])

    for i, score in enumerate(bm25_scores):
        if 0 <= i < len(chunks_df):
            combined[i] = combined.get(i, 0.0) + (1 - alpha) * float(score)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, score in ranked:
        row = chunks_df.iloc[idx]
        results.append({
            "score": round(float(score), 4),
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "act_name": row["act_name"],
            "section": row["section"],
            "section_name": row["section_name"],
            "text": row["text"],
        })
    return results

def answer_with_rag(user_query, top_k=4, mode="auto", max_new_tokens=256):
    if mode == "auto":
        mode = choose_mode(user_query)

    results = hybrid_search(user_query, top_k=top_k, mode=mode)
    context = "\n\n".join(
        [
            f"[Source {i+1}] {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}\n{r['text']}"
            for i, r in enumerate(results)
        ]
    )

    # messages = [
    #     {
    #         "role": "system",
    #         "content": (
    #             "You are an Indian legal assistant. "
    #             "BNS stands for the Bharatiya Nyaya Sanhita, 2023, the new primary criminal code of India. It came into official effect on 1 July 2024, replacing the colonial-era Indian Penal Code (IPC) of 1860. The BNS aims to modernize India's legal framework by consolidating various offences and addressing contemporary challenges like cybercrime and terrorism. "
    #             "Answer using only the provided context when possible. "
    #             "The mappings if provided in the context are officially from Government of INDIA so safe to use."
    #             "If the context is insufficient, say so clearly. "
    #             "Be concise and cite the source number(s) or section number(s) you used."
    #         )
    #     },
    #     {
    #         "role": "user",
    #         "content": f"Question: {user_query}\n\nContext:\n{context}"
    #     },
    # ]


#     messages = [
#     {
#         "role": "system",
#         "content": (
#             "You are an Indian legal assistant.\n\n"
#             "BNS means Bharatiya Nyaya Sanhita, 2023.\n"
#             "IPC means Indian Penal Code, 1860.\n\n"
#             "You must answer strictly from the retrieved context.\n"
#             "Do not guess, infer loosely, or use outside memory when the context is present.\n\n"
#             "Rules:\n"
#             "1. Prefer an exact section or subsection match over similar-looking sections.\n"
#             "2. If the question asks for an equivalent section, answer only if an exact mapping is present in the context.\n"
#             "3. Cite section numbers in natural language, not source IDs like [Source 1].\n"
#             "4. If a mapping is found, explain the mapped section briefly in 2-4 lines using the retrieved text.\n"
#             "5. If the question asks about punishment, definition, or meaning, prioritize the actual BNS or IPC section text over mapping rows.\n"
#             "6. Never confuse Section 3(1) with Section 308(3), 310(4), or any other similar number.\n"
#             "7. If multiple retrieved rows conflict, pick the best if there exist a match else if no match then avoid those context.\n\n"
#             "Output format:\n"
#             "Answer: <direct answer>\n"
#             "Section Used: <either BNS or IPC section number(s) only>\n"
#             "Explanation: <2-4 sentence explanation based only on context and the consequences of the law>\n"
#             "Note: <dont say 'No exact mapping found in retrieved context' just say 'Sorry I couldn't find anything about it, I could help anything else about Indian Law' >"
#             "  >\n"
#         )
#     },
#     {
#         "role": "user",
#         "content": f"Question: {user_query}\n\nContext:\n{context}"
#     },
# ]


    # messages = [
    # {
    # "role": "system",
    # "content": (
    #     "You are an Indian legal assistant.\n\n"

    #     "BNS refers to Bharatiya Nyaya Sanhita, 2023.\n"
    #     "IPC refers to Indian Penal Code, 1860.\n\n"

    #     "Your job is to answer legal questions clearly, correctly, and naturally.\n\n"

    #     "Strict rules:\n"
    #     "1. Always match section numbers EXACTLY. Do not reorder or approximate.\n"
    #     "   Example: 1(3) is NOT the same as 3(1).\n"
    #     "2. If the exact section is not present, do not guess or substitute similar sections.\n"
    #     "3. Do not mention words like 'context', 'mapping', 'retrieved', or 'source'.\n"
    #     "4. Do not say phrases like 'According to the provided context'.\n"
    #     "5. Do not apologize or add unnecessary notes if the answer is found.\n"
    #     "6. Prefer actual law text over summaries when available.\n\n"

    #     "Answer style:\n"
    #     "- Write a clean, natural paragraph.\n"
    #     "- Give a clear and correct explanation.\n"
    #     "- Keep it concise but informative.\n"
    #     "- At the end, include the section reference in this format:\n"
    #     "  (BNS Section X) or (IPC Section X)\n\n"

    #     "If the exact answer is not found, simply say:\n"
    #     "'I could not find a clear answer for this specific section.'\n"
    #     )
    # },
    # {
    #     "role": "user",
    #     "content": f"Question: {user_query}\n\n{context}"
    # }
    # ]

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Indian Legal Assistant. Your goal is to provide direct, authoritative, and concise answers based on the Bharatiya Nyaya Sanhita (BNS) and the Indian Penal Code (IPC).\n\n"
                "### OPERATIONAL GUIDELINES:\n"
                "1. **Tone**: Professional, direct, and helpful. Avoid meta-talk like 'Based on the context provided' or 'The mapping shows'.\n"
                "2. **Structure**: Respond in a single coherent paragraph. Start with the direct answer, followed by a brief explanation of the law's provision or consequence.\n"
                "3. **Citation**: Always end the response with a formal citation in the format: '- [Act Name] Section [Number]'.\n"
                "4. **Strict Matching**: If the user asks for a specific section (e.g., 1(3)), and the context only contains a different section (e.g., 3(1)), do NOT substitute it. State that the specific information for that section is not available.\n"
                "5. **Handling Failure**: If the retrieved context does not contain the answer, simply say: 'I'm sorry, I couldn't find the specific details regarding that section in the current legal records. Please let me know if you'd like information on a different part of Indian Law.'\n"
                "6. **No Notes**: Never include 'Note:', 'Section Used:', or 'Explanation:' headers in the final output.\n\n"
                "### EXAMPLE OF IDEAL RESPONSE:\n"
                "User: What is the punishment for murder under BNS?\n"
                "Assistant: Murder is punishable by death or imprisonment for life, and the offender shall also be liable to a fine. Specifically, when a group of five or more persons acts in concert to commit murder on grounds of race, caste, or community, every member of such group is subject to these penalties. - BNS Section 103"
            )
        },
        {
            "role": "user",
            "content": f"Use the following legal context to answer the question: \n---\n{context}\n---\nQuestion: {user_query}"
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "mode": mode,
        "answer": answer,
        "retrieved": results,
    }



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Loaded /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/full
FAISS: 1579 | Rows: 1579
Loaded /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/atomic
FAISS: 1049 | Rows: 1049


In [0]:
# ---------- example ----------
query = "IPC equivalent of BNS section 1(3)"
out = answer_with_rag(query, top_k=3, mode="atomic")

print("MODE:", out["mode"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED:")
for r in out["retrieved"]:
    print(f"- {r['score']} | {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}")

MODE: atomic

ANSWER:
 I'm sorry, I couldn't find the specific details regarding that section in the current legal records. Please let me know if you'd like information on a different part of Indian Law.

TOP RETRIEVED:
- 0.7874 | BNS_IPC_MAPPING |  | Section 6 | General explanations.
- 0.4036 | BNS_IPC_MAPPING |  | Section 385 | Putting person in fear of injury in order to commit extortion.
- 0.3943 | BNS_IPC_MAPPING |  | Section 399 | Making preparation to commit dacoity.


In [0]:

# ---------- example ----------
query = "What is the punishment for murder under BNS?"
out = answer_with_rag(query, top_k=3, mode="atomic")

print("MODE:", out["mode"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED:")
for r in out["retrieved"]:
    print(f"- {r['score']} | {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}")

MODE: atomic

ANSWER:
 Murder is punishable by death or imprisonment for life, and the offender shall also be liable to a fine. Specifically, when a group of five or more persons acts in concert to commit murder on grounds of race, caste, or community, each member of such group is subject to these penalties. - BNS Section 103 (2).

TOP RETRIEVED:
- 0.9812 | BNS_IPC_MAPPING |  | Section 303 | Punishment for murder by life- convict.
- 0.7612 | BNS_IPC_MAPPING |  | Section 307 | Attempt to murder.
- 0.5257 | BNS_IPC_MAPPING |  | Section new | Punishment for murder.


In [0]:
# ---------- example ----------
query = "What is BNS Section for IPC 420?"
out = answer_with_rag(query, top_k=3, mode="atomic")

print("MODE:", out["mode"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED:")
for r in out["retrieved"]:
    print(f"- {r['score']} | {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}")

MODE: atomic

ANSWER:
 BNS Section 318 (4) corresponds to IPC Section 420. - BNS Section 318 (4)

TOP RETRIEVED:
- 0.9704 | BNS_IPC_MAPPING |  | Section 420 | Cheating and dishonestly inducing delivery of property.
- 0.4614 | BNS_IPC_MAPPING |  | Section 416 | Cheating by personation.
- 0.4338 | BNS_IPC_MAPPING |  | Section 399 | Making preparation to commit dacoity.


In [0]:
# ---------- example ----------
query = "What is equivalent Section for BNS 3(1) in IPC?"
out = answer_with_rag(query, top_k=3, mode="atomic")

print("MODE:", out["mode"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED:")
for r in out["retrieved"]:
    print(f"- {r['score']} | {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}")

MODE: atomic

ANSWER:
 BNS 3(1) is equivalent to IPC Section 6. - IPC Section 6

TOP RETRIEVED:
- 1.0 | BNS_IPC_MAPPING |  | Section 6 | General explanations.
- 0.5726 | BNS_IPC_MAPPING |  | Section 399 | Making preparation to commit dacoity.
- 0.3541 | BNS_IPC_MAPPING |  | Section 385 | Putting person in fear of injury in order to commit extortion.


# Finetuned

In [0]:
%pip install -U transformers peft accelerate

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.9.0-py3-none-any.whl.metadata (14 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.9.0-py3-none-any.whl (637 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.41.2
    Uninstalling transformers-4.41.2:
      Successfully uninstalled transformers-4.41.2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
FINETUNED_CHECKPOINT_PATH = "./checkpoints"
USE_FINETUNED = True

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_CHECKPOINT_PATH if USE_FINETUNED else BASE_MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,   # or torch.bfloat16 if supported
    trust_remote_code=True
)

if USE_FINETUNED:
    model = PeftModel.from_pretrained(base_model, FINETUNED_CHECKPOINT_PATH)
else:
    model = base_model

model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [0]:
def generate_response(instruction, input_text=""):
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    response = full_text.split("### Response:\n")[-1].strip()
    return response


# --- Example Usage ---
query = "What is the punishment for snatching under BNS?"
print(f"\nQuery: {query}\nResponse: {generate_response(query)}")

legal_text = "Section 304 BNS: (1) Theft is snatching if..."
print(f"\nResponse: {generate_response('Explain the following legal provision.', input_text=legal_text)}")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# =========================
# BLOCK 2: INFERENCE ONLY
# =========================

import os
import json
import faiss
import torch
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ---------- config ----------
load_dotenv()

EMAIL = os.getenv("EMAIL")
HF_TOKEN = os.getenv("HF_TOKEN")

assert EMAIL, "Set EMAIL in env/.env"
assert HF_TOKEN, "Set HF_TOKEN in env/.env"

PROJECT = "nyaya_deepam"
BASE_ROOT = f"/Workspace/Users/{EMAIL}/{PROJECT}"
FULL_DIR = f"{BASE_ROOT}/artifacts/full"
ATOMIC_DIR = f"{BASE_ROOT}/artifacts/atomic"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# --------- NEW FLAGS ----------
USE_FINETUNED = True
FINETUNED_CHECKPOINT_PATH = "./checkpoints"   # change to your adapter folder

# ---------- helpers ----------
def load_index_bundle(index_dir):
    index = faiss.read_index(f"{index_dir}/corpus.faiss")

    rows = []
    with open(f"{index_dir}/chunks.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    chunks_df = pd.DataFrame(rows).reset_index(drop=True)

    if index.ntotal != len(chunks_df):
        raise ValueError(f"Mismatch in {index_dir}: FAISS={index.ntotal}, rows={len(chunks_df)}")

    texts = chunks_df["text"].fillna("").astype(str).tolist()
    bm25 = BM25Okapi([t.lower().split() for t in texts])

    print(f"Loaded {index_dir}")
    print("FAISS:", index.ntotal, "| Rows:", len(chunks_df))

    return {
        "index": index,
        "chunks_df": chunks_df,
        "bm25": bm25,
    }

def choose_mode(query: str) -> str:
    q = query.lower()
    keywords = ["equivalent", "mapping", "mapped", "difference", "changed", "compare", "ipc", "bns"]
    return "full" if any(k in q for k in keywords) else "atomic"

# ---------- login + models ----------
login(HF_TOKEN)

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

# tokenizer
TOKENIZER_SOURCE = FINETUNED_CHECKPOINT_PATH if USE_FINETUNED else LLM_MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# IMPORTANT:
# do NOT use device_map="auto" while attaching LoRA if it was causing KeyError
base_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16   # or torch.bfloat16 if supported
)

if USE_FINETUNED:
    llm_model = PeftModel.from_pretrained(base_model, FINETUNED_CHECKPOINT_PATH)
    print(f"✅ Loaded fine-tuned LoRA model from: {FINETUNED_CHECKPOINT_PATH}")
else:
    llm_model = base_model
    print(f"✅ Loaded base model only: {LLM_MODEL_NAME}")

llm_model.eval()

bundle_full = load_index_bundle(FULL_DIR)
bundle_atomic = load_index_bundle(ATOMIC_DIR)

def embed_query(query):
    return embed_model.encode(["query: " + query], normalize_embeddings=True)

def hybrid_search(query, top_k=5, alpha=0.65, mode="full"):
    bundle = bundle_full if mode == "full" else bundle_atomic
    index = bundle["index"]
    chunks_df = bundle["chunks_df"]
    bm25 = bundle["bm25"]

    q_vec = embed_query(query)
    vec_scores, vec_idx = index.search(np.array(q_vec, dtype=np.float32), top_k * 3)

    vec_scores = vec_scores[0]
    vec_idx = vec_idx[0]

    if len(vec_scores) > 0:
        vmin, vmax = vec_scores.min(), vec_scores.max()
        vec_scores = (vec_scores - vmin) / (vmax - vmin + 1e-8)

    q_tokens = query.lower().split()
    bm25_scores = np.array(bm25.get_scores(q_tokens), dtype=np.float32)

    if len(bm25_scores) > 0:
        bmin, bmax = bm25_scores.min(), bm25_scores.max()
        bm25_scores = (bm25_scores - bmin) / (bmax - bmin + 1e-8)

    combined = {}

    for i, idx in enumerate(vec_idx):
        idx = int(idx)
        if 0 <= idx < len(chunks_df):
            combined[idx] = combined.get(idx, 0.0) + alpha * float(vec_scores[i])

    for i, score in enumerate(bm25_scores):
        if 0 <= i < len(chunks_df):
            combined[i] = combined.get(i, 0.0) + (1 - alpha) * float(score)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, score in ranked:
        row = chunks_df.iloc[idx]
        results.append({
            "score": round(float(score), 4),
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "act_name": row["act_name"],
            "section": row["section"],
            "section_name": row["section_name"],
            "text": row["text"],
        })
    return results

def answer_with_rag(user_query, top_k=4, mode="auto", max_new_tokens=256):
    if mode == "auto":
        mode = choose_mode(user_query)

    results = hybrid_search(user_query, top_k=top_k, mode=mode)

    context = "\n\n".join(
        [
            f"[Source {i+1}] {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}\n{r['text']}"
            for i, r in enumerate(results)
        ]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Indian Legal Assistant. "
                "Answer only from the provided legal context. "
                "Be direct, concise, and accurate. "
                "If the answer is not clearly present, say so. "
                "End with the section citation."
            )
        },
        {
            "role": "user",
            "content": f"Use the following legal context to answer the question:\n---\n{context}\n---\nQuestion: {user_query}"
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    # move inputs to same device as model
    inputs = {k: v.to(next(llm_model.parameters()).device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "mode": mode,
        "answer": answer,
        "retrieved": results,
    }

# ---------- Example ----------
result = answer_with_rag("What is BNS Section for IPC 420?")
print(result["answer"])

print("\nRetrieved chunks:")
for r in result["retrieved"]:
    print(f"- {r['act_name']} Section {r['section']} | score={r['score']}")

---------------------------------------------------------------------------
The Python process exited with exit code 137 (SIGKILL: Killed). This may have been caused by an OOM error. Check your command's memory usage.



The last 10 KB of the process's stderr and stdout can be found below. See driver logs for full logs.
---------------------------------------------------------------------------
Last messages on stderr:
Sun Apr  5 04:23:44 2026 Connection to spark using Py4J from PID  29803
Sun Apr  5 04:23:45 2026 Initialized gateway on port 44895
Sun Apr  5 04:23:46 2026 Connected to spark using Py4J in SparkMode.REMOTE_CONNECT mode.
`torch_dtype` is deprecated! Use `dtype` instead!
Some parameters are on the meta device because they were offloaded to the disk and cpu.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-1a304597-7ff1-4832-90c2-e71018dd9bc5/lib/python3.12/site-packages/torch/nn/modules/module.py:2589: UserWarning: for base_model.model.model.layers.34.self_attn.q_proj.lora_A.def

# Key SEARCH

In [0]:
import os
import json
import re
import faiss
import torch
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- config ----------
load_dotenv()

EMAIL = os.getenv("EMAIL")
HF_TOKEN = os.getenv("HF_TOKEN")

assert EMAIL, "Set EMAIL in env/.env"
assert HF_TOKEN, "Set HF_TOKEN in env/.env"

PROJECT = "nyaya_deepam"
BASE_ROOT = f"/Workspace/Users/{EMAIL}/{PROJECT}"
FULL_DIR = f"{BASE_ROOT}/artifacts/full"
ATOMIC_DIR = f"{BASE_ROOT}/artifacts/atomic"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# direct lookup files
BNS_JSONL = "nyaya_deepam/raw/bns_rag.jsonl"
IPC_JSONL = "nyaya_deepam/raw/ipc_rag.jsonl"
MAPPING_JSONL = "nyaya_deepam/raw/bns_ipc_rag.jsonl"

# ---------- helpers ----------
def load_index_bundle(index_dir):
    index = faiss.read_index(f"{index_dir}/corpus.faiss")

    rows = []
    with open(f"{index_dir}/chunks.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    chunks_df = pd.DataFrame(rows).reset_index(drop=True)

    if index.ntotal != len(chunks_df):
        raise ValueError(f"Mismatch in {index_dir}: FAISS={index.ntotal}, rows={len(chunks_df)}")

    texts = chunks_df["text"].fillna("").astype(str).tolist()
    bm25 = BM25Okapi([t.lower().split() for t in texts])

    print(f"Loaded {index_dir}")
    print("FAISS:", index.ntotal, "| Rows:", len(chunks_df))

    return {
        "index": index,
        "chunks_df": chunks_df,
        "bm25": bm25,
    }

def choose_mode(query: str) -> str:
    q = query.lower()
    keywords = ["equivalent", "mapping", "mapped", "difference", "changed", "compare", "ipc", "bns"]
    return "full" if any(k in q for k in keywords) else "atomic"

# ---------- exact lookup helpers ----------
def normalize_act_name(act_text: str):
    if not act_text:
        return None
    t = act_text.lower().strip()
    if "bns" in t or "bharatiya nyaya sanhita" in t:
        return "BNS"
    if "ipc" in t or "indian penal code" in t:
        return "IPC"
    return None

def normalize_section_number(section_text: str):
    if not section_text:
        return None
    return str(section_text).strip().lower()

def load_jsonl_records(jsonl_path):
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def build_section_lookup(records):
    lookup = {}

    for row in records:
        metadata = row.get("metadata", {})
        text = row.get("text", "")

        act_name_raw = metadata.get("act_name", "") or text
        act = normalize_act_name(act_name_raw)

        section_raw = metadata.get("section", "")
        section = normalize_section_number(section_raw)

        if act and section:
            lookup[(act, section)] = row

    return lookup

def build_mapping_lookup(records):
    mapping_lookup = {}

    for row in records:
        meta = row.get("metadata", {})
        ipc = str(meta.get("ipc_section", "")).strip().lower()
        bns = str(meta.get("bns_section_subsection", "")).strip().lower()

        if ipc and bns:
            mapping_lookup[ipc] = row

    return mapping_lookup

def extract_act_and_section_from_query(query: str):
    q = query.lower().strip()

    bns_match = re.search(r'\b(?:section|sec\.?)\s+(\d+[a-z]?(?:\(\d+\))?)\s*(?:of\s*)?bns\b', q)
    ipc_match = re.search(r'\b(?:section|sec\.?)\s+(\d+[a-z]?(?:\(\d+\))?)\s*(?:of\s*)?ipc\b', q)

    # if both mentioned, ignore exact lookup
    if bns_match and ipc_match:
        return None, None
    if bns_match:
        return "BNS", bns_match.group(1).lower()
    if ipc_match:
        return "IPC", ipc_match.group(1).lower()

    return None, None

def smart_lookup(query, section_lookup, mapping_lookup):
    act, section = extract_act_and_section_from_query(query)

    if act is None or section is None:
        return None

    if act == "BNS":
        direct = section_lookup.get(("BNS", section))
        if direct:
            return {
                "type": "direct_bns",
                "act": "BNS",
                "section": section,
                "text": direct["text"],
                "raw": direct
            }

    if act == "IPC":
        direct = section_lookup.get(("IPC", section))
        if direct:
            return {
                "type": "direct_ipc",
                "act": "IPC",
                "section": section,
                "text": direct["text"],
                "raw": direct
            }

        mapping = mapping_lookup.get(section)
        if mapping:
            return {
                "type": "mapping",
                "act": "IPC",
                "section": section,
                "bns_equivalent": mapping["metadata"].get("bns_section_subsection", ""),
                "text": mapping["text"],
                "raw": mapping
            }

    return None

def build_exact_context(exact_match):
    if exact_match is None:
        return ""

    if exact_match["type"] == "direct_bns":
        return (
            f"[Exact Match]\n"
            f"Act: BNS\n"
            f"Section: {exact_match['section']}\n"
            f"{exact_match['text']}\n"
        )

    if exact_match["type"] == "direct_ipc":
        return (
            f"[Exact Match]\n"
            f"Act: IPC\n"
            f"Section: {exact_match['section']}\n"
            f"{exact_match['text']}\n"
        )

    if exact_match["type"] == "mapping":
        return (
            f"[Exact Mapping Match]\n"
            f"IPC Section: {exact_match['section']}\n"
            f"BNS Equivalent: {exact_match['bns_equivalent']}\n"
            f"{exact_match['text']}\n"
        )

    return ""

# ---------- login + models ----------
login(HF_TOKEN)

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto"
)

bundle_full = load_index_bundle(FULL_DIR)
bundle_atomic = load_index_bundle(ATOMIC_DIR)

# ---------- load direct lookup data ----------
bns_records = load_jsonl_records(BNS_JSONL)
ipc_records = load_jsonl_records(IPC_JSONL)
mapping_records = load_jsonl_records(MAPPING_JSONL)

section_lookup = build_section_lookup(bns_records + ipc_records)
mapping_lookup = build_mapping_lookup(mapping_records)

def embed_query(query):
    return embed_model.encode(["query: " + query], normalize_embeddings=True)

def hybrid_search(query, top_k=5, alpha=0.65, mode="full"):
    bundle = bundle_full if mode == "full" else bundle_atomic
    index = bundle["index"]
    chunks_df = bundle["chunks_df"]
    bm25 = bundle["bm25"]

    q_vec = embed_query(query)
    vec_scores, vec_idx = index.search(np.array(q_vec, dtype=np.float32), top_k * 3)

    vec_scores = vec_scores[0]
    vec_idx = vec_idx[0]

    if len(vec_scores) > 0:
        vmin, vmax = vec_scores.min(), vec_scores.max()
        vec_scores = (vec_scores - vmin) / (vmax - vmin + 1e-8)

    q_tokens = query.lower().split()
    bm25_scores = np.array(bm25.get_scores(q_tokens), dtype=np.float32)

    if len(bm25_scores) > 0:
        bmin, bmax = bm25_scores.min(), bm25_scores.max()
        bm25_scores = (bm25_scores - bmin) / (bmax - bmin + 1e-8)

    combined = {}

    for i, idx in enumerate(vec_idx):
        idx = int(idx)
        if 0 <= idx < len(chunks_df):
            combined[idx] = combined.get(idx, 0.0) + alpha * float(vec_scores[i])

    for i, score in enumerate(bm25_scores):
        if 0 <= i < len(chunks_df):
            combined[i] = combined.get(i, 0.0) + (1 - alpha) * float(score)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, score in ranked:
        row = chunks_df.iloc[idx]
        results.append({
            "score": round(float(score), 4),
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "act_name": row["act_name"],
            "section": row["section"],
            "section_name": row["section_name"],
            "text": row["text"],
        })
    return results

def answer_with_rag(user_query, top_k=4, mode="auto", max_new_tokens=256):
    if mode == "auto":
        mode = choose_mode(user_query)

    # 1. exact lookup first
    exact_match = smart_lookup(user_query, section_lookup, mapping_lookup)
    exact_context = build_exact_context(exact_match)

    # 2. normal rag
    results = hybrid_search(user_query, top_k=top_k, mode=mode)

    rag_context = "\n\n".join(
        [
            f"[Source {i+1}] {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}\n{r['text']}"
            for i, r in enumerate(results)
        ]
    )

    # 3. combine
    if exact_context.strip():
        context = exact_context + "\n\n" + rag_context
    else:
        context = rag_context

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert Indian Legal Assistant. Your goal is to provide direct, authoritative, and concise answers based on the Bharatiya Nyaya Sanhita (BNS) and the Indian Penal Code (IPC).\n\n"
                "### OPERATIONAL GUIDELINES:\n"
                "1. **Tone**: Professional, direct, and helpful. Avoid meta-talk like 'Based on the context provided' or 'The mapping shows'.\n"
                "2. **Structure**: Respond in a single coherent paragraph. Start with the direct answer, followed by a brief explanation of the law's provision or consequence.\n"
                "3. **Citation**: Always end the response with a formal citation in the format: '- [Act Name] Section [Number]'.\n"
                "4. **Strict Matching**: If the user asks for a specific section (e.g., 1(3)), and the context only contains a different section (e.g., 3(1)), do NOT substitute it. State that the specific information for that section is not available.\n"
                "5. **Handling Failure**: If the retrieved context does not contain the answer, simply say: 'I'm sorry, I couldn't find the specific details regarding that section in the current legal records. Please let me know if you'd like information on a different part of Indian Law.'\n"
                "6. **No Notes**: Never include 'Note:', 'Section Used:', or 'Explanation:' headers in the final output.\n"
                "7. **MAPPING**:If user asked about IPC and there is information about BNS, provide BNS section and All details about it.\n"
                "8. **Provide more details**: If user asked about IPC or BNS, provide entire details we have about that IPC or BNS section.\n"
                "9. **Section Citation Exception**: If Section number of IPC or BNS is mentioned already in the context, do not mention it again at the end.\n\n"
                "### EXAMPLE OF IDEAL RESPONSE:\n"
                "#### EXAMPLE 1::"
                "User: What is the punishment for murder under BNS?\n"
                "Assistant: Murder is punishable by death or imprisonment for life, and the offender shall also be liable to a fine. Specifically, when a group of five or more persons acts in concert to commit murder on grounds of race, caste, or community, every member of such group is subject to these penalties. - BNS Section 103"
                "#### EXAMPLE 2::\n"
                "User: What is Section 33 IPC?\n"
                "Assistant: Section 33 of the IPC defines both 'act' and 'omission.' It states that an 'act' may mean either a single act or a series of acts, and an “omission” may mean either a single omission or a series of omissions. In the BNS, this concept has been split, with 'act' covered under Section 2(1) and 'omission' under Section 2(25).\n"
                

            )
        },
        {
            "role": "user",
            "content": f"Use the following legal context to answer the question: \n---\n{context}\n---\nQuestion: {user_query}"
        },
    ]


    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "mode": mode,
        "answer": answer,
        "retrieved": results,
        "exact_match": exact_match,
        "used_exact_match": exact_match is not None,
    }

  

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/full
FAISS: 1579 | Rows: 1579
Loaded /Workspace/Users/mahanthyalla@iisc.ac.in/nyaya_deepam/artifacts/atomic
FAISS: 1049 | Rows: 1049


In [0]:
# ---------- example ----------
query = "What is BNS Section for IPC 420?"
out = answer_with_rag(query, top_k=3, mode="atomic")

print("MODE:", out["mode"])
print("\nANSWER:\n", out["answer"])
print("\nTOP RETRIEVED:")
for r in out["retrieved"]:
    print(f"- {r['score']} | {r['source']} | {r['act_name']} | Section {r['section']} | {r['section_name']}")

IOStream.flush timed out


MODE: atomic

ANSWER:
 BNS Section 318 (4) corresponds to IPC Section 420. - BNS Section 318 (4)

TOP RETRIEVED:
- 0.9704 | BNS_IPC_MAPPING |  | Section 420 | Cheating and dishonestly inducing delivery of property.
- 0.4614 | BNS_IPC_MAPPING |  | Section 416 | Cheating by personation.
- 0.4338 | BNS_IPC_MAPPING |  | Section 399 | Making preparation to commit dacoity.


In [0]:
out

{'mode': 'atomic',
 'answer': 'BNS Section 318 (4) corresponds to IPC Section 420. - BNS Section 318 (4)',
 'retrieved': [{'score': 0.9704,
   'chunk_id': 'BNS_318_(4)_IPC_420',
   'source': 'BNS_IPC_MAPPING',
   'act_name': '',
   'section': '420',
   'section_name': 'Cheating and dishonestly inducing delivery of property.',
   'text': 'BNS-IPC Mapping\n\nBNS Section/Subsection: 318 (4)\nIPC Section: 420\nSubject: Cheating and dishonestly inducing delivery of property.\n\nMapping:\nBNS 318 (4) ↔ IPC 420\n\nChanges:\nIPC section is included as a sub-section in BNS sans heading.'},
  {'score': 0.4614,
   'chunk_id': 'BNS_319_(1)_IPC_416',
   'source': 'BNS_IPC_MAPPING',
   'act_name': '',
   'section': '416',
   'section_name': 'Cheating by personation.',
   'text': 'BNS-IPC Mapping\n\nBNS Section/Subsection: 319 (1)\nIPC Section: 416\nSubject: Cheating by personation.\n\nMapping:\nBNS 319 (1) ↔ IPC 416\n\nChanges:\nFormal change. This is a definition section which is presented as a sub

In [0]:
answer_with_rag("What is Section 419 IPC?",mode="atomic")   

{'mode': 'atomic',
 'answer': 'Section 419 of the IPC deals with cheating by personation and includes it as a sub-section within BNS Section 319 (2). The punishment for this offense is imprisonment for a term which may extend to five years, along with a fine. - IPC Section 419',
 'retrieved': [{'score': 0.8344,
   'chunk_id': 'BNS_319_(2)_IPC_419',
   'source': 'BNS_IPC_MAPPING',
   'act_name': '',
   'section': '419',
   'section_name': 'Cheating by personation punishment.',
   'text': 'BNS-IPC Mapping\n\nBNS Section/Subsection: 319 (2)\nIPC Section: 419\nSubject: Cheating by personation punishment.\n\nMapping:\nBNS 319 (2) ↔ IPC 419\n\nChanges:\nIPC section is included as a sub-section in BNS sans heading. Upper limit of imprisonment is increased from three years to five years.'},
  {'score': 0.7519,
   'chunk_id': 'BNS_319_(1)_IPC_416',
   'source': 'BNS_IPC_MAPPING',
   'act_name': '',
   'section': '416',
   'section_name': 'Cheating by personation.',
   'text': 'BNS-IPC Mapping\n

In [0]:
def print_result(query, result):
    print("="*80)
    print(f"🧠 QUERY: {query}")
    print("-"*80)

    print(f"📌 Mode: {result['mode']}")
    print(f"✅ Used Exact Match: {result['used_exact_match']}")
    
    if result["exact_match"]:
        print("\n🔍 EXACT MATCH:")
        em = result["exact_match"]
        print(f"Type: {em['type']}")
        print(f"Act: {em.get('act', '')}")
        print(f"Section: {em.get('section', '')}")
        if em["type"] == "mapping":
            print(f"BNS Equivalent: {em.get('bns_equivalent', '')}")
        print("\nText:")
        print(em["text"][:500], "...")  # truncate

    print("\n📚 RETRIEVED CONTEXT (Top Chunks):")
    for i, r in enumerate(result["retrieved"]):
        print(f"\n[{i+1}] {r['act_name']} Section {r['section']} | Score: {r['score']}")
        print(r["text"][:300], "...")

    print("\n💡 FINAL ANSWER:")
    print(result["answer"])
    print("="*80 + "\n")


queries = [
    "What is Section 420 IPC?",
    "Explain Section 96 BNS in simple terms",
    "What does Section 33 IPC define?",
    "What is punishment for cheating under IPC?",
    "Explain Section 2(1) BNS",
    "Difference between IPC Section 95 and BNS equivalent",
    "What is Section 378 IPC and its BNS mapping?",
    "Explain abetment under IPC with section",
    "What is Section 304 BNS punishment?",
    "Compare IPC Section 24 and BNS provisions"
]


for q in queries:
    result = answer_with_rag(q, mode="atomic")
    print_result(q, result)

🧠 QUERY: What is Section 420 IPC?
--------------------------------------------------------------------------------
📌 Mode: atomic
✅ Used Exact Match: True

🔍 EXACT MATCH:
Type: mapping
Act: IPC
Section: 420
BNS Equivalent: 318 (4)

Text:
BNS-IPC Mapping

BNS Section/Subsection: 318 (4)
IPC Section: 420
Subject: Cheating and dishonestly inducing delivery of property.

Mapping:
BNS 318 (4) ↔ IPC 420

Changes:
IPC section is included as a sub-section in BNS sans heading. ...

📚 RETRIEVED CONTEXT (Top Chunks):

[1]  Section 420 | Score: 1.0
BNS-IPC Mapping

BNS Section/Subsection: 318 (4)
IPC Section: 420
Subject: Cheating and dishonestly inducing delivery of property.

Mapping:
BNS 318 (4) ↔ IPC 420

Changes:
IPC section is included as a sub-section in BNS sans heading. ...

[2]  Section 416 | Score: 0.3079
BNS-IPC Mapping

BNS Section/Subsection: 319 (1)
IPC Section: 416
Subject: Cheating by personation.

Mapping:
BNS 319 (1) ↔ IPC 416

Changes:
Formal change. This is a definition sectio